# 13. Comparing HAR and GARCH (and the Diebold–Mariano test)
Both benchmarks are ready. Which one is more accurate — overall and in each crisis? And is the difference **real**, or could it be luck? That second question is answered by the **Diebold–Mariano test**, built here step by step.

* **Needs:** `work/har_forecasts.csv`, `work/garch_forecasts.csv`, `work/dataset.csv`. **Makes:** nothing new.

*Simple notebook series of the project 'This Time Is Different?'. Prepared with AI assistance (declared in `notes/ai_use.md`); a study notebook, not dissertation text.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
folder = '/content/drive/MyDrive/DataSets/ThisTimeIsDifferent/'

import warnings
warnings.filterwarnings('ignore')     # hide warning messages

In [2]:
from pandas import read_csv, DataFrame
har = read_csv(folder + 'work/har_forecasts.csv', index_col=0, parse_dates=True)
garch = read_csv(folder + 'work/garch_forecasts.csv', index_col=0, parse_dates=True)
dataset = read_csv(folder + 'work/dataset.csv', index_col=0, parse_dates=True)

results = DataFrame({'rv5_fwd': dataset.loc[har.index, 'rv5_fwd'],
                     'HAR': har['HAR'],
                     'GARCH': garch['GARCH'],
                     'Naive': dataset.loc[har.index, 'rv_w'] * 5,
                     'window': dataset.loc[har.index, 'window']})
results.head()

,rv5_fwd,HAR,GARCH,Naive,window
2007-01-02,0.000289,0.000363,0.000292,0.000305,calm
2007-01-03,0.000208,0.000364,0.000323,0.000443,calm
2007-01-04,0.000197,0.000356,0.000310,0.000425,calm
2007-01-05,0.000186,0.000350,0.000295,0.000369,calm
2007-01-08,0.000118,0.000362,0.000314,0.000455,calm


## QLIKE for every day and every model

In [3]:
from numpy import log
ratio = results['rv5_fwd'] / results['HAR']
results['QLIKE_HAR'] = ratio - log(ratio) - 1
ratio = results['rv5_fwd'] / results['GARCH']
results['QLIKE_GARCH'] = ratio - log(ratio) - 1
ratio = results['rv5_fwd'] / results['Naive']
results['QLIKE_Naive'] = ratio - log(ratio) - 1
results[['QLIKE_HAR', 'QLIKE_GARCH', 'QLIKE_Naive']].mean().round(3)

QLIKE_HAR      0.414
QLIKE_GARCH    0.380
QLIKE_Naive    0.802
dtype: float64

Over all 4,230 test days GARCH has the lowest QLIKE (0.380), then HAR (0.414). In each window:

In [4]:
results.groupby('window')[['QLIKE_HAR', 'QLIKE_GARCH', 'QLIKE_Naive']].mean().round(3)

,QLIKE_HAR,QLIKE_GARCH,QLIKE_Naive
window,,,
Brexit referendum,1.973,1.941,1.264
COVID-19,0.763,0.700,1.053
Global Financial Crisis,0.676,0.401,0.834
Irish sovereign debt crisis,0.380,0.374,0.802
War / energy shock 2022,0.282,0.290,0.506
calm,0.360,0.351,0.802


HAR is clearly worse in the GFC (0.676 vs 0.401): it was trained only on the calm 2003–06 years and reacted too slowly. In the Irish crisis the two are almost equal.

## The Diebold–Mariano test, step by step
**Question:** is the average difference in QLIKE between HAR and GARCH really different from zero?

In [5]:
d = results['QLIKE_HAR'] - results['QLIKE_GARCH']      # step 1: daily difference in loss
n = len(d)
d_mean = d.mean()                                      # step 2: average difference
print('n =', n, '| average difference =', round(d_mean, 4))

n = 4230 | average difference = 0.0342


**Step 3 — how much does `d` vary?** Neighbouring days are related (the 5-day targets overlap), so besides the normal variance we also add how each day relates to the 1, 2, 3 and 4 days before it (*autocovariances*):

In [6]:
dc = d - d_mean
gamma0 = (dc * dc).sum() / n
gamma1 = (dc * dc.shift(1)).sum() / n
gamma2 = (dc * dc.shift(2)).sum() / n
gamma3 = (dc * dc.shift(3)).sum() / n
gamma4 = (dc * dc.shift(4)).sum() / n
long_run_variance = gamma0 + 2 * (gamma1 + gamma2 + gamma3 + gamma4)
print('long-run variance =', long_run_variance)

long-run variance = 0.42849391905949663


**Step 4 — the test statistic**, like a t-statistic: the average divided by its standard error, with a small-sample correction (Harvey, Leybourne & Newbold, *HLN*):

In [7]:
from numpy import sqrt
dm = d_mean / sqrt(long_run_variance / n)
h = 5
dm_hln = dm * sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
print('DM statistic =', round(dm_hln, 2))

DM statistic = 3.39


**Step 5 — the p-value** from the t distribution: the probability of a difference this large if the two models were really equally good.

In [8]:
from scipy.stats import t
p_value = 2 * t.sf(abs(dm_hln), df=n - 1)
print('p-value =', round(p_value, 4))

p-value = 0.0007


p = 0.0007 < 0.05: GARCH is **significantly** more accurate than HAR (the average difference is positive = HAR's loss is higher).

## The same steps in a function
We will need this test again (notebooks 18 and 24), so the steps go into one function — nothing new, just the five steps above:

In [9]:
def dm_test(loss_a, loss_b):
    d = loss_a - loss_b
    n = len(d)
    d_mean = d.mean()
    dc = d - d_mean
    long_run_variance = (dc * dc).sum() / n
    for k in [1, 2, 3, 4]:
        long_run_variance = long_run_variance + 2 * (dc * dc.shift(k)).sum() / n
    dm = d_mean / sqrt(long_run_variance / n)
    dm_hln = dm * sqrt((n + 1 - 10 + 20 / n) / n)
    p_value = 2 * t.sf(abs(dm_hln), df=n - 1)
    print('average difference =', round(d_mean, 4), '| DM =', round(dm_hln, 2), '| p-value =', round(p_value, 4))


dm_test(results['QLIKE_HAR'], results['QLIKE_GARCH'])

average difference = 0.0342 | DM = 3.39 | p-value = 0.0007


With MSE instead of QLIKE (×1,000,000 to make the numbers readable):

In [10]:
results['MSE_HAR'] = (results['rv5_fwd'] - results['HAR']) ** 2 * 1000000
results['MSE_GARCH'] = (results['rv5_fwd'] - results['GARCH']) ** 2 * 1000000
dm_test(results['MSE_HAR'], results['MSE_GARCH'])

average difference = 0.6419 | DM = 1.53 | p-value = 0.1261
